In [ ]:
"""
Script Name: Science Community Values and Self-Efficacy Visualization (Figures 6 & 7). These scores are measured using the instrument of Tripartite Integration Model of Social Influence (Estrada et al., 2011) to answer our research questions measuring for three constructs: self-efficacy, science identity, and scientific community values 
Description: Generates a plot using Plotly Express to visualize 
             pre- and post-test shifts in Science Community Values (Fig 6) and Self-Efficacy (Fig 7).
             Connects pre/post dots with lines dynamically colored by the magnitude 
             of the score difference.
Repository: scienceintegration-change-visualization
Publication: https://link.springer.com/article/10.1186/s40594-026-00608-z
"""

# ==========================================
# 0. PACKAGES TO INSTALL
# ==========================================

import pandas as pd
import plotly.express as px

# ==========================================
# 1. DATA LOADING & PREPARATION
# ==========================================

# NOTE: Replace with your actual data file path
# df = pd.read_excel("your_data_sheet.xlsx")

# Sort the dataframe by the interested change/difference metric, then by the baseline/initial score
# This structures the data to create a clean, progressive gradient in the final plot
df_sorted = df.sort_values(by=['Score_Difference', 'Pre_Score'], ascending=[True, True])

# ==========================================
# 2. BASE PLOT CREATION (using Plotly Express)
# ==========================================

# Create a scatter plot representing Pre and Post values on the X-axis against IDs on the Y-axis
fig = px.scatter(
    df_sorted, 
    x=['Pre_Score', 'Post_Score'], 
    y='ID', 
    labels={'variable': ''}, 
    width=1000, 
    height=2000
)

# Customize the layout for publication quality
fig.update_layout(
    title_text="",  # Add a title here if desired
    yaxis_title="",
    xaxis_title="Measurement Values",  # Change this to your specific metric label
    margin_l=200                       # Padding on the left for Y-axis labels
)

# Increase marker size for better visibility
fig.update_traces(marker=dict(size=10))

# Hide Y-axis tick labels to protect participant anonymity and reduce visual clutter 
fig.update_yaxes(showticklabels=False)


# ==========================================
# 3. LINKING THE LINES TO GENERATE THE PLOT
# ==========================================

# Map specific ranges of score differences to shades of gray/colors. These were the ranges I used in the publication. 
# Larger shifts use darker colors; zero/no-change shifts use orange.
color_ranges = {
    (1, 6): 'rgb(30, 30, 30)',
    (0.5, 1): 'rgb(80, 80, 80)',
    (0.1, 0.5): 'rgb(130, 130, 130)',
    (-0.5, 0.1): 'rgb(130, 130, 130)',
    (-1, -0.5): 'rgb(80, 80, 80)',
    (-6, -1): 'rgb(30, 30, 30)',
    (0, 0): 'orange'
}

# Iterate through each row in the dataset to draw connecting lines between Pre and Post dots
for i in range(df_sorted.shape[0]):
    score_diff = df_sorted['Score_Difference'].iloc[i]
    
    # Identify which color range the current row's score difference falls into
    for range_, color in color_ranges.items():
        if range_[0] < score_diff <= range_[1]:
            line_color = color
            break
    else:
        # Fallback case if no range matches
        continue

    # Draw a horizontal line connecting the Pre_Score dot to the Post_Score dot for this ID
    fig.add_shape(
        type='line',
        x0=df_sorted['Pre_Score'].iloc[i], y0=df_sorted['ID'].iloc[i],
        x1=df_sorted['Post_Score'].iloc[i], y1=df_sorted['ID'].iloc[i],
        line_color=line_color
    )


# ==========================================
# 4. AXIS CONFIGURATION & DISPLAY
# ==========================================

# Dynamically set X-axis limits based on the maximum score in the dataset to prevent edge-clipping
max_x_val = max(df_sorted['Pre_Score'].max(), df_sorted['Post_Score'].max())
fig.update_xaxes(range=[0.5, max_x_val + 0.5])

# Render the final interactive plot
fig.show()